# Retrieval-guided master-prompt selection

## 1. Goal and single-split contract

This notebook compares configured master prompts while holding retrieval and inference controls explicit. One run evaluates exactly one configured split:

- `validation` maps to the Bias-in-Bios `dev` source;
- `test` maps to the Bias-in-Bios `test` source;
- the unselected split may appear in descriptive source counts, but it is never embedded, predicted, scored, ranked, or plotted.

Every configured condition is evaluated on the selected split. Conditions are ranked separately within each language model, and exactly one row per language model receives `is_best=True`. There is no automatic transition from validation to test.

## 2. Environment and imports

Run this notebook from the repository root in the `prompt-selection` Conda environment. The setup cell enables IPython autoreload so edits to local `.py` modules are applied before later cells run. Imports use the same modules as `app.py`; there are no standalone dataset downloads or direct model probes.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, JSON, Markdown, display

from dataset import (
    calculate_dataset_counts,
    load_source_rows,
    select_profession_splits,
    select_run_data,
    task_settings,
)
from evaluation import resolve_metric_column
from modeling import build_prompt
from configuration import load_config, validate_config
from pipeline import prepare_embedding_cache, run_experiment

In [3]:
PROJECT_ROOT = Path.cwd().resolve()
CONFIG_PATH = PROJECT_ROOT / 'config.yaml'

if not CONFIG_PATH.exists():
    raise FileNotFoundError('Open this notebook from the repository root.')

## 3. Load and validate the configuration

`config.yaml` is the single source of experiment settings. Each `prompt_templates` entry contains the complete master-prompt wording, including `{labels}`; the code only substitutes placeholders and does not append prompt text. The dataset and models use required pinned revisions. `inference.prediction_method` selects one run-wide method: deterministic `generated_output` with a 32-token output limit and lowercase exact-label validation, or `log_probability` scoring. Validation rejects unknown enums, invalid scalar types, empty values, duplicates, unsupported prompt placeholders, and inconsistent pool sizes before model work begins.

In [4]:
config = load_config(CONFIG_PATH)
validate_config(config)

evaluation_split = config['defaults']['evaluation_split']
prediction_method = config['inference']['prediction_method']
target, audit_column, professions, target_labels = task_settings(config)

display(Markdown(
    f'**Configured run:** hold out `{target}` and evaluate `{evaluation_split}`; '
    f'the language model receives `hard_text + {audit_column}` and uses '
    f'`prediction_method: {prediction_method}`.'
))

**Configured run:** hold out `profession` and evaluate `validation`; the language model receives `hard_text + gender` and uses `prediction_method: generated_output`.

## 4. Condition count and runtime controls

For each language model and prompt template, zero contributes one retrieval-independent condition when configured. Positive example counts are crossed with retrieval methods, embedding models, and example orders. Total predictions also multiply by language-model count and selected evaluation rows. Zero-shot rows record those three retrieval controls as `not_applicable`.

The main row multiplier is `evaluation_per_profession_gender`. A positive integer fixes every selected profession-gender cell at that size. `max_balanced` resolves after source loading to the smallest available cell in the configured evaluation split, preserving equal cell sizes. The flow prints the resolved integer. Loaded language-model context overflows fail explicitly; embedding inputs exceeding their configured sequence limit are reported and truncated by the encoder.

`balanced_semantic` preserves its greedy balanced-selection sequence internally so every configured example-count prefix remains balanced. After the requested prefix is sliced, `most_similar_first` and `most_similar_last` stably sort only that fixed set by retrieval score for prompt presentation; `shuffle` deterministically permutes the same set.

In [5]:
retrieval = config['retrieval']
prompt_count = len(config['prompt_templates'])
positive_example_count_count = sum(
    example_count > 0 for example_count in retrieval['example_counts']
)
zero_shot_conditions_per_model = (
    prompt_count if 0 in retrieval['example_counts'] else 0
)
few_shot_conditions_per_model = (
        len(retrieval['methods'])
        * len(retrieval['embedding_models'])
        * positive_example_count_count
        * len(retrieval['example_orders'])
        * prompt_count
)
conditions_per_model = (
    zero_shot_conditions_per_model + few_shot_conditions_per_model
)
language_model_count = len(config['inference']['language_models'])
evaluation_setting = config['dataset']['evaluation_per_profession_gender']
evaluation_row_count = (
    len(professions) * 2 * evaluation_setting
    if isinstance(evaluation_setting, int)
    else 'resolved after source loading'
)
total_label_predictions = (
    language_model_count * conditions_per_model * evaluation_row_count
    if isinstance(evaluation_row_count, int)
    else evaluation_row_count
)

pd.DataFrame([{
    'language_models': language_model_count,
    'zero_shot_conditions_per_language_model': zero_shot_conditions_per_model,
    'few_shot_conditions_per_language_model': few_shot_conditions_per_model,
    'conditions_per_language_model': conditions_per_model,
    'total_conditions': language_model_count * conditions_per_model,
    'evaluation_per_profession_gender': evaluation_setting,
    'selected_evaluation_rows': evaluation_row_count,
    'total_label_predictions': total_label_predictions,
}])

,language_models,zero_shot_conditions_per_language_model,few_shot_conditions_per_language_model,conditions_per_language_model,total_conditions,evaluation_per_profession_gender,selected_evaluation_rows,total_label_predictions
0,4,0,48,48,192,5,40,7680


## 5. Load all sources, select run rows, and inspect composition

`load_source_rows()` loads every normalized source row and `select_profession_splits()` creates the canonical keys `train`, `validation`, and `test`. `select_run_data()` is the only step that applies the train cap and chooses balanced evaluation cells.

The source table is descriptive. The run table is authoritative for rows used by retrieval and evaluation.

In [6]:
source_rows = load_source_rows(config, PROJECT_ROOT)
source_splits = select_profession_splits(config, source_rows)
train_rows, evaluation_rows, resolved_evaluation_per_cell = select_run_data(config, source_splits)

display(Markdown(
    f'**Resolved evaluation rows per profession-gender cell:** '
    f'`{resolved_evaluation_per_cell}`'
))

Selecting evaluation cells:   0%|          | 0/8 [00:00<?, ?cell/s]

**Resolved evaluation rows per profession-gender cell:** `5`

In [7]:
source_dataset_counts = calculate_dataset_counts(config, source_splits)

display(Markdown('### Full filtered source composition'))
source_dataset_counts

### Full filtered source composition

,split,profession,gender,count,gender_share_within_profession,profession_share_within_gender,cell_share_of_split,gender_share_gap_within_profession,profession_share_gap_within_gender
0,train,professor,male,42130,0.548939,0.560009,0.306344,0.097879,0.473010
1,train,professor,female,34618,0.451061,0.555720,0.251722,0.097879,0.452740
2,train,physician,male,13492,0.506304,0.179341,0.098106,0.012609,0.473010
3,train,physician,female,13156,0.493696,0.211192,0.095663,0.012609,0.452740
4,train,attorney,male,13064,0.617129,0.173652,0.094994,0.234258,0.473010
5,train,attorney,female,8105,0.382871,0.130109,0.058935,0.234258,0.452740
6,train,journalist,male,6545,0.505015,0.086999,0.047591,0.010031,0.473010
7,train,journalist,female,6415,0.494985,0.102979,0.046646,0.010031,0.452740
8,validation,professor,male,6482,0.548950,0.560000,0.306318,0.097900,0.473002
9,validation,professor,female,5326,0.451050,0.555602,0.251689,0.097900,0.452535


In [8]:
run_dataset_counts = calculate_dataset_counts(config, {'train': train_rows, evaluation_split: evaluation_rows})

display(Markdown('### Rows selected for this run'))
run_dataset_counts

### Rows selected for this run

,split,profession,gender,count,gender_share_within_profession,profession_share_within_gender,cell_share_of_split,gender_share_gap_within_profession,profession_share_gap_within_gender
0,train,professor,male,42130,0.548939,0.560009,0.306344,0.097879,0.47301
1,train,professor,female,34618,0.451061,0.555720,0.251722,0.097879,0.45274
2,train,physician,male,13492,0.506304,0.179341,0.098106,0.012609,0.47301
3,train,physician,female,13156,0.493696,0.211192,0.095663,0.012609,0.45274
4,train,attorney,male,13064,0.617129,0.173652,0.094994,0.234258,0.47301
5,train,attorney,female,8105,0.382871,0.130109,0.058935,0.234258,0.45274
6,train,journalist,male,6545,0.505015,0.086999,0.047591,0.010031,0.47301
7,train,journalist,female,6415,0.494985,0.102979,0.046646,0.010031,0.45274
8,validation,professor,male,5,0.500000,0.250000,0.125000,0.000000,0.00000
9,validation,professor,female,5,0.500000,0.250000,0.125000,0.000000,0.00000


In [9]:
assert {row['split'] for row in evaluation_rows} == {evaluation_split}
assert set(run_dataset_counts['split']) == {'train', evaluation_split}
assert set(run_dataset_counts.loc[run_dataset_counts['split'].eq(evaluation_split), 'count']) == {resolved_evaluation_per_cell}

## 6. Preview the language-model input

This preview shows the exact zero-shot format when zero is configured and the maximum positive-count format when present. It uses the first selected query and representative train rows; during the run, retrieval chooses condition-specific demonstrations. Each actual message list is saved in the prediction table's `prompt` column.

In [10]:
preview_template_name, preview_template = next(iter(config['prompt_templates'].items()))
configured_example_counts = config['retrieval']['example_counts']
positive_preview_counts = [
    example_count for example_count in configured_example_counts if example_count > 0
]
preview_example_counts = ([0] if 0 in configured_example_counts else [])
if positive_preview_counts:
    preview_example_counts.append(max(positive_preview_counts))

for preview_example_count in preview_example_counts:
    preview_messages = build_prompt(
        evaluation_rows[0],
        train_rows[:preview_example_count],
        target,
        target_labels,
        preview_template,
    )
    display(Markdown(
        f'**Template:** `{preview_template_name}`; **Examples:** `{preview_example_count}`'
    ))
    display(JSON(preview_messages, expanded=True))

**Template:** `neutral`; **Examples:** `8`

<IPython.core.display.JSON object>

## 7. Prepare the complete training-embedding cache

This cell prepares each configured, revision-pinned embedding model when any positive example count is configured and automatically skips preparation for an all-zero run. Training rows are stably sorted from longest to shortest before bounded chunks are embedded and stored; each row's `train_order` retains its canonical source position. A complete table is reused on later calls; changes to professions, `train_size`, evaluation size, query prompt, or embedding batch size do not rebuild it.

In [11]:
has_positive_example_count = any(
    example_count > 0 for example_count in config['retrieval']['example_counts']
)
prepared_embedding_rows = (
    prepare_embedding_cache(config, PROJECT_ROOT, progress=print)
    if has_positive_example_count
    else {}
)
prepared_embedding_rows

Using device: mps
Preparing 257478 canonical training rows


Preparing complete embedding caches:   0%|          | 0/2 [00:00<?, ?model/s]

Reusing 257478 complete training embeddings from /Users/AmirMohammad/Documents/Prompt Selection (B.Sc. Project)/retrieval-guided-master-prompt-selection/data/lancedb/semantic_qwen_qwen3-embedding-8b
Reusing 257478 complete training embeddings from /Users/AmirMohammad/Documents/Prompt Selection (B.Sc. Project)/retrieval-guided-master-prompt-selection/data/lancedb/semantic_baai_bge-large-en-v1.5
Prepared complete embedding caches at /Users/AmirMohammad/Documents/Prompt Selection (B.Sc. Project)/retrieval-guided-master-prompt-selection/data/lancedb


{'Qwen/Qwen3-Embedding-8B': 257478, 'BAAI/bge-large-en-v1.5': 257478}

## 8. Run the experiment

When a language model CSV is missing, this cell requires a complete, current training-embedding cache. After each language model finishes, its raw predictions are saved atomically under `<output_dir>/incomplete_run/`. Running this cell again reuses each available model CSV, regenerates all metrics and final artifacts, and deletes the checkpoint after complete success. The cache is intentionally not compared with the current YAML, so run `discard_incomplete_run(config, PROJECT_ROOT)` before changing experiment settings.

In [12]:
# from pipeline import discard_incomplete_run
#
# discard_incomplete_run(config, PROJECT_ROOT)

In [13]:
run = run_experiment(config, PROJECT_ROOT, progress=print)

Incomplete-run checkpoints: /Users/AmirMohammad/Documents/Prompt Selection (B.Sc. Project)/retrieval-guided-master-prompt-selection/results/incomplete_run
Holding out profession; language model input is hard_text + gender
Prediction method: generated_output


Selecting evaluation cells:   0%|          | 0/8 [00:00<?, ?cell/s]

Loaded 211585 filtered source rows; selected 137525 train and 40 validation rows for this run


Evaluating language models on validation:   0%|          | 0/4 [00:00<?, ?model/s]

Using device: mps


...

Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

Evaluating Qwen/Qwen3.5-27B:   0%|          | 0/48 [00:00<?, ?condition/s]

Predicting validation rows:   0%|          | 0/40 [00:00<?, ?row/s]

Predicting validation rows:   0%|          | 0/40 [00:00<?, ?row/s]

KeyboardInterrupt: 

In [ ]:
assert run['evaluation_split'] == evaluation_split
assert set(run['predictions']['evaluation_split']) == {evaluation_split}
resumed_model_text = ', '.join(run['resumed_language_models']) or 'none'
display(Markdown(
    f'Artifacts written to `{run["run_dir"]}`. Resumed models: `{resumed_model_text}`.'
))

## 9. Inspect rankings and current-split winners

Ranks restart at 1 for each language model. `is_best` identifies one winner per model, chosen using the configured metric and direction on this run's split.

In [ ]:
metric_column = resolve_metric_column(config['defaults']['ranking_metric'])
ranking_columns = ['evaluation_split', 'language_model', 'rank', 'is_best', 'condition', metric_column]
ranking_columns = [column for column in ranking_columns if column in run['results'].columns]
run['results'][ranking_columns]

In [ ]:
best_conditions = run['results'].loc[run['results']['is_best']].copy()
assert len(best_conditions) == len(config['inference']['language_models'])
display(Markdown('### One current-split winner per language model'))
best_conditions[ranking_columns]

### Compare one experimental factor across matched conditions

Set `comparison_factor` to any condition factor. Every other factor is held fixed, and only complete matched blocks containing every value of the selected factor are retained. `condition` is not a matching key because its text embeds the selected factor.

In [ ]:
comparison_results_path = run['run_dir'] / f'{evaluation_split}_results.csv'
comparison_results = pd.read_csv(comparison_results_path)

# Change only this value to compare another experimental factor.
comparison_factor = 'language_model'

condition_factor_columns = [
    'language_model',
    'retrieval_method',
    'embedding_model',
    'example_count',
    'example_order',
    'prompt_name',
]
comparison_context_columns = [
    'evaluation_split',
    'target',
    'audit_column',
]

if comparison_factor not in condition_factor_columns:
    raise ValueError(
        f'{comparison_factor!r} must be one of {condition_factor_columns}'
    )
if comparison_results[comparison_factor].isna().any():
    raise ValueError(f'{comparison_factor!r} contains missing values')

comparison_factor_levels = sorted(
    comparison_results[comparison_factor].unique().tolist()
)
if len(comparison_factor_levels) < 2:
    raise ValueError(
        f'{comparison_factor!r} must contain at least two values'
    )

comparison_fixed_columns = comparison_context_columns + [
    column
    for column in condition_factor_columns
    if column != comparison_factor
]
comparison_key = comparison_fixed_columns + [comparison_factor]

if comparison_results.duplicated(comparison_key).any():
    raise ValueError(
        f'Multiple rows exist for the same comparison key: {comparison_key}'
    )

has_every_factor_level = (
    comparison_results
    .groupby(comparison_fixed_columns, dropna=False)[comparison_factor]
    .transform('nunique')
    .eq(len(comparison_factor_levels))
)

matched_condition_results = (
    comparison_results.loc[has_every_factor_level]
    .sort_values(comparison_key, kind='stable')
    .reset_index(drop=True)
)
if matched_condition_results.empty:
    raise ValueError(
        f'No complete matched blocks exist for {comparison_factor!r}'
    )

matched_condition_results.insert(
    0,
    'comparison_group',
    (
        matched_condition_results
        .groupby(comparison_fixed_columns, sort=False, dropna=False)
        .ngroup()
        + 1
    ),
)

comparison_display_columns = [
    'comparison_group',
    *comparison_fixed_columns,
    comparison_factor,
    metric_column,
]
matched_condition_results[comparison_display_columns]

### Aggregate the selected factor's paired influence

The default decision set keeps six complementary metrics: macro F1 for overall quality, worst audit-group accuracy for the performance floor, minimum demographic-parity ratio for worst-label selection parity, audit-group accuracy difference for the overall group gap, and mean/maximum equalized-odds differences for typical and worst-label error-rate disparity. They remain separate rather than forming one composite score. `mean_delta_from_baseline` first subtracts the baseline inside each complete matched block and then averages those paired differences. Higher-is-better metrics appear first, followed by lower-is-better metrics. The code includes a commented one-line switch for restoring every numeric performance and fairness metric.

In [ ]:
excluded_numeric_columns = {
    'comparison_group',
    'rank',
    'is_best',
    'sample_count',
    'n_target_labels',
    'n_audit_groups',
    *condition_factor_columns,
}
all_comparison_metric_columns = [
    column
    for column in matched_condition_results.select_dtypes(
        include='number'
    ).columns
    if column not in excluded_numeric_columns
    and not column.startswith('n_')
]

# Focused decision set: quality, group robustness, and complementary fairness views.
comparison_metric_columns = [
    'macro_f1',
    'worst_audit_group_accuracy',
    'min_demographic_parity_ratio',
    'audit_group_accuracy_difference',
    'mean_equalized_odds_difference',
    'max_equalized_odds_difference',
]

# Uncomment this line to include every numeric performance and fairness metric.
# comparison_metric_columns = all_comparison_metric_columns.copy()

missing_comparison_metrics = sorted(
    set(comparison_metric_columns) - set(all_comparison_metric_columns)
)
if missing_comparison_metrics:
    raise ValueError(
        f'Comparison metrics are missing from the results: '
        f'{missing_comparison_metrics}'
    )

higher_is_better_metric_columns = [
    column
    for column in comparison_metric_columns
    if not column.endswith('_difference')
]
lower_is_better_metric_columns = [
    column
    for column in comparison_metric_columns
    if column.endswith('_difference')
]
ordered_comparison_metric_columns = (
    higher_is_better_metric_columns
    + lower_is_better_metric_columns
)

comparison_metric_values = matched_condition_results.melt(
    id_vars=['comparison_group', comparison_factor],
    value_vars=ordered_comparison_metric_columns,
    var_name='metric',
    value_name='value',
)

comparison_baseline = comparison_factor_levels[0]
comparison_baseline_values = (
    comparison_metric_values.loc[
        comparison_metric_values[comparison_factor].eq(comparison_baseline)
    ]
    .set_index(['comparison_group', 'metric'])['value']
    .rename('baseline_value')
)
comparison_metric_values = comparison_metric_values.join(
    comparison_baseline_values,
    on=['comparison_group', 'metric'],
)
comparison_metric_values['delta_from_baseline'] = (
    comparison_metric_values['value']
    - comparison_metric_values['baseline_value']
)

comparison_metric_summary = (
    comparison_metric_values
    .groupby(
        ['metric', comparison_factor],
        as_index=False,
        dropna=False,
        sort=False,
    )
    .agg(
        mean=('value', 'mean'),
        std=('value', 'std'),
        mean_delta_from_baseline=('delta_from_baseline', 'mean'),
        n_defined=('value', 'count'),
    )
)
comparison_metric_sort_order = {
    metric: index
    for index, metric in enumerate(ordered_comparison_metric_columns)
}
comparison_metric_summary['_metric_sort_order'] = (
    comparison_metric_summary['metric'].map(comparison_metric_sort_order)
)
comparison_metric_summary.insert(
    1,
    'direction',
    comparison_metric_summary['metric'].isin(
        higher_is_better_metric_columns
    ).map({True: 'higher is better', False: 'lower is better'}),
)
comparison_metric_summary = (
    comparison_metric_summary
    .sort_values(
        ['_metric_sort_order', comparison_factor],
        kind='stable',
    )
    .drop(columns='_metric_sort_order')
    .reset_index(drop=True)
)
comparison_metric_summary

## 10. Inspect detailed metrics and plots

CSV metric tables retain every condition. The bounded previews below focus on winning conditions. Summary plots rank all conditions; detailed target-label, audit-group, fairness, coverage, and confusion plots focus on the winners.

In [ ]:
best_names = set(best_conditions['condition'])
for table_name in ('target_label_metrics', 'audit_group_metrics', 'fairness_metrics', 'confusion_matrix'):
    table = run[table_name]
    display(Markdown(f'### {table_name.replace("_", " ").title()}'))
    display(table.loc[table['condition'].isin(best_names)].head(200))

In [ ]:
display(Markdown('### Plot files'))
for plot_name, plot_path in run['plots'].items():
    display(Markdown(f'**{plot_name.replace("_", " ").title()}**'))
    display(Image(filename=str(plot_path)))

## 11. Inspect best prompts and bounded prediction previews

The text report contains only this split's winners and ranking score. Prediction previews show true label, audit group, chosen label, condition metadata, prediction method, and either generated model output or allowed-label scores; the complete table remains in the split-prefixed CSV.

In [ ]:
display(Markdown(run['best_prompts'].read_text(encoding='utf-8')))

prediction_detail_columns = [
    column for column in ['prediction_method', 'model_output', 'label_scores']
    if column in run['predictions'].columns
]
prediction_columns = [
    'evaluation_split', 'language_model', 'query_id', 'true_label',
    'audit_group', 'predicted_label',
] + prediction_detail_columns + ['condition']
best_predictions = run['predictions'].loc[
    run['predictions']['condition'].isin(best_names), prediction_columns,
]
display(best_predictions.head(200))

## 12. Metric reference and next-run guidance

### 1. Notation, counts, and supports

For one prediction condition, $N$ is `sample_count`, $K$ is `n_target_labels`,
and $G$ is `n_audit_groups`. Row $i$ has true target label $y_i$, predicted
target label $\hat y_i$, and audit group $a_i$. Target label $c$ is evaluated
one-vs-rest: $c$ is positive and every other target label is negative. Audit
group $g$ contains $N_g$ rows, stored as `audit_group_n`.

$$
\begin{aligned}
TP_c&=\sum_i\mathbf{1}[y_i=c\land\hat y_i=c], &
FP_c&=\sum_i\mathbf{1}[y_i\ne c\land\hat y_i=c],\\
FN_c&=\sum_i\mathbf{1}[y_i=c\land\hat y_i\ne c], &
TN_c&=\sum_i\mathbf{1}[y_i\ne c\land\hat y_i\ne c].
\end{aligned}
$$

These four counts are stored as `tp`, `fp`, `fn`, and `tn`. The remaining
detailed output columns are:

- `positive_support`: $n_c=TP_c+FN_c$;
- `negative_support`: $FP_c+TN_c=N-n_c$;
- `predicted_positive`: $TP_c+FP_c$;
- `selection_rate`: $SR_c=(TP_c+FP_c)/N$.

The audit-group table applies exactly the same definitions after restricting
the sums to $a_i=g$; its counts use the subscript $(c,g)$ and its selection-rate
denominator is $N_g$. The long-form confusion-matrix output stores `count` as

$$
C_{r,s}=\sum_i\mathbf{1}[y_i=r\land\hat y_i=s]
$$

for every configured true-label row $r$ and predicted-label column $s$.

### 2. Target-label and audit-group rates

For each target label, and identically within each audit group:

- `precision`: $PPV_c=TP_c/(TP_c+FP_c)$;
- `recall`: $TPR_c=TP_c/(TP_c+FN_c)$;
- `f1`: $F1_c=2TP_c/(2TP_c+FP_c+FN_c)$;
- `specificity`: $TNR_c=TN_c/(TN_c+FP_c)$;
- `false_positive_rate`: $FPR_c=FP_c/(FP_c+TN_c)$;
- `false_negative_rate`: $FNR_c=FN_c/(FN_c+TP_c)$;
- `negative_predictive_value`: $NPV_c=TN_c/(TN_c+FN_c)$.

`audit_group_accuracy` is the multiclass accuracy repeated for each target-label
row of the same audit group:

$$
Accuracy_g=\frac{\sum_{i:a_i=g}\mathbf{1}[y_i=\hat y_i]}{N_g}.
$$

Whenever their denominators are nonzero, $FPR=1-Specificity$ and
$FNR=1-Recall$. Higher selection rate is not inherently better or worse;
precision, recall, F1, specificity, NPV, and audit-group accuracy are better
when higher, while FPR and FNR are better when lower.

The implementation returns `NaN` for a rate whose required denominator is
zero. Macro and weighted aggregates omit undefined rates; a coverage output
reports how many values remained. Selection rate and audit-group accuracy are
defined because validated conditions and observed audit groups are nonempty.

### 3. Overall quality and agreement

For any target-label rate $m_c$, let $D_m$ contain the target labels where it is
defined, and let $D_m^+$ additionally require positive true support $n_c>0$:

$$
\begin{aligned}
Accuracy&=\frac{\sum_cTP_c}{N},\\
Macro(m)&=\frac{1}{|D_m|}\sum_{c\in D_m}m_c,\\
Weighted(m)&=\frac{\sum_{c\in D_m^+}n_cm_c}
{\sum_{c\in D_m^+}n_c}.
\end{aligned}
$$

These formulas map to `macro_precision`, `macro_recall / balanced_accuracy`,
`macro_f1`, `weighted_precision`, and `weighted_f1`. The defined-label coverage
outputs are
`n_precision_defined_target_labels` $=|D_{Precision}|$,
`n_recall_defined_target_labels` $=|D_{Recall}|$, and
`n_f1_defined_target_labels` $=|D_{F1}|$. An aggregate is `NaN` when its defined
set is empty; a weighted aggregate also needs at least one positive weight.

In this single-label multiclass task, let $T=\sum_cTP_c$ be the number of
correct rows and $E=\sum_cFP_c=\sum_cFN_c$ the number of errors. Since
$N=T+E$:

$$
MicroPrecision=MicroRecall=MicroF1=Accuracy=\frac{T}{N},
$$

$$
WeightedRecall=\frac{1}{N}\sum_{c:n_c>0}n_c\frac{TP_c}{n_c}
=Accuracy,\qquad
BalancedAccuracy=\frac{1}{|D_{Recall}|}\sum_{c\in D_{Recall}}Recall_c
=MacroRecall.
$$

The results store those equality families once under
`accuracy / micro_precision / micro_recall / micro_f1 / weighted_recall` and
`macro_recall / balanced_accuracy`. Each individual standard name is still a
valid `ranking_metric` alias.

For multiclass agreement, let $C$ be the confusion matrix,
$s=\sum_{r,k}C_{r,k}=N$, $q=\operatorname{trace}(C)$,
$p_k=\sum_rC_{r,k}$ be predicted totals, and $t_k=\sum_rC_{k,r}$ be true totals:

The `matthews_correlation_coefficient` output is

$$
MCC=\frac{qs-\sum_kp_kt_k}
{\sqrt{(s^2-\sum_kp_k^2)(s^2-\sum_kt_k^2)}}.
$$

With observed agreement $p_o=Accuracy$ and chance-expected agreement
$p_e=\sum_k(t_k/s)(p_k/s)$:

The `cohen_kappa` output is

$$
\kappa=\frac{p_o-p_e}{1-p_e}.
$$

Higher accuracy, MCC, and kappa are better; one means perfect agreement. The
implementation uses scikit-learn's degenerate-case conventions: MCC is `0.0`
when its denominator is zero, while kappa is `NaN` when $1-p_e=0$.

### 4. Target-label fairness across audit groups

For target label $c$ and rate $m$, let $G_{m,c}$ contain the audit groups where
$m_{c,g}$ is defined. A range ignores undefined values and exists only when at
least two values remain:

$$
Range_g(m_{c,g})=\max_{g\in G_{m,c}}m_{c,g}
-\min_{g\in G_{m,c}}m_{c,g}.
$$

The `fairness_metrics` columns are:

- `demographic_parity_difference`: $DPDiff_c=Range_g(SR_{c,g})$;
- `demographic_parity_ratio`:
  $DPRatio_c=\min_{g\in G_{SR,c}}SR_{c,g}/\max_{g\in G_{SR,c}}SR_{c,g}$;
- `equal_opportunity_difference`: $EODiff_c=Range_g(TPR_{c,g})$;
- `false_positive_rate_difference`: $FPRDiff_c=Range_g(FPR_{c,g})$;
- `equalized_odds_difference`: $EOddsDiff_c=\max(EODiff_c,FPRDiff_c)$;
- `predictive_parity_difference`: $PPDiff_c=Range_g(PPV_{c,g})$.

A difference of zero and a demographic-parity ratio of one mean equality across
the compared audit groups. The ratio requires at least two defined selection
rates and a positive maximum; if every group has zero selection rate, it is
`NaN` rather than $0/0$. Equalized-odds difference is `NaN` unless both its TPR-
and FPR-range components are defined.

The per-target-label coverage outputs are:

- `n_audit_groups_compared` $=G$;
- `n_selection_rate_defined_audit_groups` $=|G_{SR,c}|$;
- `n_recall_defined_audit_groups` $=|G_{TPR,c}|$;
- `n_false_positive_rate_defined_audit_groups` $=|G_{FPR,c}|$;
- `n_precision_defined_audit_groups` $=|G_{PPV,c}|$.

### 5. Condition-level fairness summaries and coverage

Audit-group accuracy is summarized as:

- `worst_audit_group_accuracy`: $\min_gAccuracy_g$;
- `audit_group_accuracy_difference`: $Range_g(Accuracy_g)$.

The accuracy difference requires at least two audit groups. Higher worst-group
accuracy and a lower accuracy difference are preferable.

For each target-label fairness value $d_c$, let $D_d$ contain the target labels
where it is defined. Every fairness family receives the same three summaries:

$$
Mean(d)=\frac{1}{|D_d|}\sum_{c\in D_d}d_c,\qquad
Min(d)=\min_{c\in D_d}d_c,\qquad
Max(d)=\max_{c\in D_d}d_c.
$$

The exact result-column families are:

| Per-target-label value $d$       | Condition result columns                                                                                          |
|----------------------------------|-------------------------------------------------------------------------------------------------------------------|
| `demographic_parity_difference`  | `mean_demographic_parity_difference`, `min_demographic_parity_difference`, `max_demographic_parity_difference`    |
| `demographic_parity_ratio`       | `mean_demographic_parity_ratio`, `min_demographic_parity_ratio`, `max_demographic_parity_ratio`                   |
| `equal_opportunity_difference`   | `mean_equal_opportunity_difference`, `min_equal_opportunity_difference`, `max_equal_opportunity_difference`       |
| `false_positive_rate_difference` | `mean_false_positive_rate_difference`, `min_false_positive_rate_difference`, `max_false_positive_rate_difference` |
| `equalized_odds_difference`      | `mean_equalized_odds_difference`, `min_equalized_odds_difference`, `max_equalized_odds_difference`                |
| `predictive_parity_difference`   | `mean_predictive_parity_difference`, `min_predictive_parity_difference`, `max_predictive_parity_difference`       |

Each family omits undefined target-label values. If $D_d$ is empty, all three
summaries are `NaN`. Its exact coverage output is:

- `n_demographic_parity_defined_target_labels` $=|D_{DPDiff}|$;
- `n_demographic_parity_ratio_defined_target_labels` $=|D_{DPRatio}|$;
- `n_equal_opportunity_defined_target_labels` $=|D_{EODiff}|$;
- `n_false_positive_rate_defined_target_labels` $=|D_{FPRDiff}|$;
- `n_equalized_odds_defined_target_labels` $=|D_{EOddsDiff}|$;
- `n_predictive_parity_defined_target_labels` $=|D_{PPDiff}|$.

For difference metrics, zero is best, so their minimum is the best target-label
value and their maximum is the worst. For demographic-parity ratio, one is best,
so the minimum ratio is the worst target-label value and the maximum is the best.
Coverage plots show $K$ or $G$ first so every defined count has an explicit
denominator.

Quality and fairness must be interpreted together. With `target: profession`
and audit column `gender`, these are conventional protected-group fairness
diagnostics. With `target: gender` and audit column `profession`, the same
mathematics is better described as profession-conditioned performance or
stereotype/association diagnostics. Profession and gender remain separate
experiments with the same prompt candidates and separate winners.

### Next-run guidance

Keep `evaluation_split: validation` while developing prompt candidates. Change it deliberately to `test` only when you intend to evaluate the complete condition grid on test. Profession and gender are separate experiments with the same prompt candidates and separate winners; do not mix their rankings.